# scPoli reference + Ishikawa query (Object A)

* **Developed by:** Anna Maguza
* **Affilation:** CellZome, a GSK company
* **Created date:** 2026-05-08
* **Last modified date:** 2026-05-08

scarches scPoli for reference-query mapping.


## obj_a_scpoli

Object A — model scpoli (label-aware on cell_states).

**Kernel: `scarches_env`**.

In [ ]:
import os
import sys
import types
import scanpy as sc, anndata as ad, numpy as np, torch, scvi
import matplotlib.pyplot as plt
from pathlib import Path
DATA_OUT = Path('/Users/am336941/PhD/data/Fetal_stem_cells_analysis_enhanced')

In [2]:
INPUT = Path('/Users/am336941/PhD/data/Fetal_stem_cells_analysis_enhanced/integrated_obj_a_scanvi.h5ad')
adata = sc.read_h5ad(INPUT)
adata

AnnData object with n_obs × n_vars = 102013 × 2000
    obs: 'organism', 'organism_part', 'cell_type', 'sample_id', 'Study_name', 'library_preparation_protocol', 'age_group', 'celltype', 'cell_states', 'gut_region', 'lgr5_status', 'species', 'source', 'n_genes_by_counts', 'total_counts', '_scvi_batch', '_scvi_labels'
    var: 'gene_id', 'gene_name', 'gene_type', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: 'Study_name_colors', '_scvi_manager_uuid', '_scvi_uuid', 'age_group_colors', 'cell_states_colors', 'gut_region_colors', 'hvg', 'lgr5_status_colors', 'neighbors', 'processing_history', 'umap'
    obsm: 'X_scanvi', 'X_scvi', 'X_umap', '_scvi_extra_categorical_covs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [3]:
REPO     = Path('/Users/am336941/Library/CloudStorage/OneDrive-GSK/Desktop/Fetal_stem_cells_analysis')
FIG_DIR  = REPO / 'analysis_enhanced/3_integration/3a_human_only/figures/obj_a_scarches'
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
def _accelerator():
    try:
        import torch
        return 'mps' if torch.backends.mps.is_available() else 'cpu'
    except Exception:
        return 'cpu'
ACCEL = _accelerator()
print('using accelerator:', ACCEL)


using accelerator: mps


In [5]:
scvi_model = scvi.model.SCVI.load(f"/Users/am336941/PhD/data/Fetal_stem_cells_analysis_enhanced/model_scvi_obj_a_scvi", adata)

INFO     File /Users/am336941/PhD/data/Fetal_stem_cells_analysis_enhanced/model_scvi_obj_a_scvi/model.pt already   
         downloaded                                                                                                


/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scvi/model/base/_base_model.py:862: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator='mps' in train function.In future releases it will become default for mps supported machines.
  _, _, device = parse_device_args(


In [ ]:
#scvi.model.SCVI.setup_anndata(adata, layer='counts', batch_key='Study_name',
#                              categorical_covariate_keys=['species'] if 'species' in adata.obs.columns else None)


In [12]:
for _mod in ['scarches.plotting.scvi_eval', 'scarches.plotting.trvae_eval']:
    if _mod not in sys.modules:
        _dummy = types.ModuleType(_mod)
        _dummy.SCVI_EVAL = None
        _dummy.TRVAE_EVAL = None
        sys.modules[_mod] = _dummy

from scarches.models.scpoli import scPoli


In [13]:
m = scPoli(adata, condition_keys=['Study_name'], cell_type_keys=['cell_states'],
           latent_dim=30, embedding_dims=10)

Embedding dictionary:
 	Num conditions: [3]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 2000 45 10
	Mean/Var Layer in/out: 45 30
Decoder Architecture:
	First Layer in, out and cond:  30 45 10
	Output Layer in/out:  45 2000 



/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scarches/models/scpoli/scpoli_model.py:151: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  self.obs_metadata_ = adata.obs.groupby('conditions_combined').first()
/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scarches/models/scpoli/scpoli_model.py:156: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  self.obs_metadata_ = adata.obs.groupby(condition_keys).first()
/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scarches/models/scpoli/scpoli_model.py:164: FutureWarning: Series.__getitem__ treati

In [14]:
m.train(n_epochs=150, pretraining_epochs=30)

INFO:scarches.trainers.scpoli.trainer:GPU available: False


Initializing dataloaders
Starting training
 |███████████---------| 59.3%  - val_loss:  760.39 - val_cvae_loss:  743.31 - val_prototype_loss:   17.08 - val_labeled_loss:   17.08
ADJUSTED LR
 |████████████████----| 80.0%  - val_loss:  760.69 - val_cvae_loss:  742.61 - val_prototype_loss:   18.07 - val_labeled_loss:   18.07
ADJUSTED LR
 |█████████████████---| 89.3%  - val_loss:  761.09 - val_cvae_loss:  742.95 - val_prototype_loss:   18.14 - val_labeled_loss:   18.14
ADJUSTED LR
 |██████████████████--| 94.0%  - val_loss:  760.88 - val_cvae_loss:  742.76 - val_prototype_loss:   18.12 - val_labeled_loss:   18.12
Stopping early: no improvement of more than 0 nats in 20 epochs
If the early stopping criterion is too strong, please instantiate it with different parameters in the train method.


In [15]:
adata.obsm['X_scpoli'] = m.get_latent(adata, mean=True)

In [16]:
sc.pp.neighbors(adata, use_rep='X_scpoli', n_neighbors=50, metric='minkowski')
sc.tl.umap(adata, random_state=0, min_dist=0.2, spread=1.0)

/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/neighbors/__init__.py:430: FutureWarning: The method obsm_keys is deprecated and will be removed in the future. Use obsm instead of obsm_keys. (e.g. `k in adata.obsm` or `adata.obsm.keys() | {'u'}`)
  if "X_diffmap" in adata.obsm_keys():


In [17]:
COLOR_KEYS = ['Study_name', 'cell_states', 'age_group', 'lgr5_status', 'gut_region']
QC_KEYS    = ['total_counts', 'n_genes_by_counts']

In [20]:
sc.set_figure_params(dpi=300, figsize=(6, 6))
sc.pl.umap(adata, color=[c for c in COLOR_KEYS if c in adata.obs.columns],
           cmap='magma_r', ncols=3, size=2, frameon=False, show=False)
plt.savefig(FIG_DIR / 'umap_metadata.png', bbox_inches='tight'); plt.close()

sc.pl.umap(adata, color=QC_KEYS, cmap='magma_r', ncols=2, size=2,
           frameon=False, show=False)
plt.savefig(FIG_DIR / 'umap_qc.png', bbox_inches='tight'); plt.close()
print('UMAPs saved \u2192', FIG_DIR)

UMAPs saved → /Users/am336941/Library/CloudStorage/OneDrive-GSK/Desktop/Fetal_stem_cells_analysis/analysis_enhanced/3_integration/3a_human_only/figures/obj_a_scarches


In [22]:
try:
    from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
    bio = BioConservation(isolated_labels=True, nmi_ari_cluster_labels_kmeans=True,
                          silhouette_label=True, clisi_knn=True)
    batch = BatchCorrection(graph_connectivity=True, ilisi_knn=True,
                            kbet_per_label=False, pcr_comparison=True, bras=True)
    a_lab = adata[adata.obs['cell_states'].astype(str) != 'unknown'].copy()
    bm = Benchmarker(a_lab, batch_key='Study_name', label_key='cell_states',
                     embedding_obsm_keys=['X_scpoli','X_scvi', 'X_pca'],
                     bio_conservation_metrics=bio, batch_correction_metrics=batch, n_jobs=2)
    bm.benchmark()
    res = bm.get_results(min_max_scale=False, clean_names=True)
    print(res)
    res.to_csv(FIG_DIR / 'scib_metrics.csv')
    plt.figure(figsize=(10, 6))
    bm.plot_results_table(min_max_scale=False, show=False)
    plt.savefig(FIG_DIR / 'scib_benchmark.png', dpi=300, bbox_inches='tight')
    plt.close()
except Exception as e:
    print('scIB skipped:', e)

/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(adata, mask_var, use_highly_variable)
/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scanpy/preprocessing/_pca/__init__.py:243: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  Version(ad.__version__) < Version("0.9")
Embeddings:   0%|          | 0/3 [00:00<?, ?it/s]/Users/am336941/uv_envs/lgr5_enhanced/.venv/lib/python3.13/site-packages/scib_metrics/metrics/_graph_connectivity.py:32: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  tab = p

              Isolated labels        KMeans NMI        KMeans ARI  \
Embedding                                                           
X_scpoli             0.704342          0.290668          0.185434   
X_scvi               0.563557          0.264208          0.157273   
X_pca                0.587344          0.038401          0.015442   
Metric Type  Bio conservation  Bio conservation  Bio conservation   

             Silhouette label             cLISI              BRAS  \
Embedding                                                           
X_scpoli             0.453625          0.966002          0.732608   
X_scvi                0.50486          0.911424          0.927911   
X_pca                 0.41283          0.923039          0.641541   
Metric Type  Bio conservation  Bio conservation  Batch correction   

                        iLISI Graph connectivity    PCR comparison  \
Embedding                                                            
X_scpoli                  0.0 

<Figure size 3000x1800 with 0 Axes>

In [23]:
adata.write_h5ad(DATA_OUT / 'integrated_obj_a_scpoli.h5ad', compression='gzip')
m.save(str(DATA_OUT / 'model_scpoli_obj_a_scpoli'))